# Coordinates Cities

## Dengue

In [6]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm

In [7]:
# read files
folder_path = "Dengue"  # Adjust this to your folder path

files_dengue = [file_path for file_path in glob.glob(os.path.join(folder_path, "*")) if os.path.isfile(file_path)]

In [ ]:
# Assuming files_dengue is a list of file paths to Excel files
dataframes = []

cols_to_keep = [
    'SEMANA', 'ANO', 'Estado_final_de_caso',
    'Pais_ocurrencia', 'Departamento_ocurrencia', 'Municipio_ocurrencia'
]

for file in tqdm(files_dengue):
    try:
        df_temp = pd.read_excel(file)
        df_temp = df_temp[cols_to_keep]
        df_temp = df_temp[df_temp['Estado_final_de_caso'] != 2]

        # Group by desired columns and count occurrences
        df_group = (
            df_temp.groupby(['SEMANA', 'ANO', 'Pais_ocurrencia', 'Departamento_ocurrencia', 'Municipio_ocurrencia'])
            .size()
            .reset_index(name='count')
        )

        # Create idx_city column if needed
        df_group['idx_city'] = (
            df_group['Pais_ocurrencia'].astype(str) + '_' +
            df_group['Departamento_ocurrencia'].astype(str) + '_' +
            df_group['Municipio_ocurrencia'].astype(str)
        )

        dataframes.append(df_group)

    except Exception as e:
        print(f"Error reading {file}: {e}")

  6%|▌         | 1/17 [00:05<01:26,  5.41s/it]

In [9]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(dataframes, ignore_index=True)

In [10]:
df

,SEMANA,ANO,idx_city,count
0,1,2007,COLOMBIA_ARAUCA_ARAUCA,2
1,1,2007,COLOMBIA_ARAUCA_SARAVENA,1
2,1,2007,COLOMBIA_ATLANTICO_BARANOA,7
3,1,2007,COLOMBIA_ATLANTICO_BARRANQUILLA,23
4,1,2007,COLOMBIA_ATLANTICO_GALAPA,2
...,...,...,...,...
147043,52,2023,COLOMBIA_VAUPES_MITU,8
147044,52,2023,COLOMBIA_VICHADA_PUERTO CARREÑO,1
147045,52,2023,COMORAS_EXTERIOR_EXTERIOR_COMORAS,1
147046,52,2023,HAITÍ_EXTERIOR_EXTERIOR_HAITÍ,1


# Coordinates

In [13]:
# split the idx_city column into separate columns
df[['Country', 'Department', 'City']] = df['idx_city'].str.split('_', expand=True)

ValueError: Columns must be same length as key

In [11]:
from geopy.geocoders import Nominatim
import folium
import time
import re
from unidecode import unidecode
import matplotlib.pyplot as plt

In [12]:
# Initialize geocoder
geolocator = Nominatim(user_agent="weather_locator")

In [ ]:
# Function to clean text
def clean_text(text):
    text = str(text).lower()               # Lowercase
    text = unidecode(text)                 # Remove accents
    text = re.sub(r'[^\w\s]', '', text)    # Remove punctuation
    text = re.sub(r'\s+', ' ', text)       # Normalize whitespace
    return text.strip()

# Apply to relevant columns
df['City'] = df['City'].apply(clean_text)
df['Departamento'] = df['Departamento'].apply(clean_text)
df['Country'] = df['Country'].apply(clean_text)

# Optional: remove duplicate cities (if any)
df.drop_duplicates(subset=['City', 'Departamento', 'Country'], inplace=True)